In [1]:
import os
import csv
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

In [2]:
# -----------------------
# CONFIG
# -----------------------
MODEL_PATHS = [
    "cnn_model_0.h5",
    "cnn_model_1.h5",
    "cnn_model_2.h5",
    "cnn_model_3.h5",
    "cnn_model_4.h5",
]

WEIGHTS = np.array([1.0, 1.0, 1.2, 1.0, 1.5])
WEIGHTS = WEIGHTS / WEIGHTS.sum()

ENSEMBLE_THRESHOLD = 0.45

DATASET_ROOT = "./dataset_new"
OUTPUT_CSV = "ensemble_predictions.csv"
OUTPUT_XLSX = "ensemble_predictions.xlsx"
IMG_SIZE = (224, 224)

# Excel styles
GREEN_FILL = PatternFill("solid", fgColor="C6EFCE")
RED_FILL = PatternFill("solid", fgColor="FFC7CE")
HEADER_FILL = PatternFill("solid", fgColor="BDD7EE")

In [3]:
# -----------------------
# HELPERS
# -----------------------
def preprocess(path):
    img = load_img(path, target_size=IMG_SIZE)
    arr = img_to_array(img) / 255.0
    return np.expand_dims(arr, axis=0)

def ensemble_predict(models, img):
    probs = []
    for m in models:
        p = m.predict(img, verbose=0)
        p = float(p[0][0]) if p.shape[-1] == 1 else float(p[0][1])
        probs.append(p)
    probs = np.array(probs)
    return float(np.sum(probs * WEIGHTS))

In [4]:
# -----------------------
# MAIN
# -----------------------
def main():
    print("Loading models...")
    models = [load_model(p, compile=False) for p in MODEL_PATHS]

    images = []
    for cls in ["normal", "osteoporosis"]:
        folder = os.path.join(DATASET_ROOT, cls)
        for f in os.listdir(folder):
            if f.lower().endswith((".png", ".jpg", ".jpeg")):
                images.append((os.path.join(folder, f), cls))

    wb = Workbook()
    ws = wb.active
    ws.title = "Ensemble Results"

    header = ["Image", "True Class", "Ensemble Prob", "Prediction", "Status"]
    ws.append(header)
    for i in range(1, len(header)+1):
        ws.cell(row=1, column=i).fill = HEADER_FILL
        ws.column_dimensions[get_column_letter(i)].width = 25

    csv_rows = [header]
    correct = 0

    for img_path, true_cls in images:
        img = preprocess(img_path)
        prob = ensemble_predict(models, img)

        pred = "osteoporosis" if prob >= ENSEMBLE_THRESHOLD else "normal"
        status = "Correct" if pred == true_cls else "Wrong"
        correct += status == "Correct"

        ws.append([os.path.basename(img_path), true_cls, prob, pred, status])
        fill = GREEN_FILL if status == "Correct" else RED_FILL
        for c in range(1, 6):
            ws.cell(row=ws.max_row, column=c).fill = fill

        csv_rows.append([img_path, true_cls, prob, pred, status])

    with open(OUTPUT_CSV, "w", newline="") as f:
        csv.writer(f).writerows(csv_rows)

    wb.save(OUTPUT_XLSX)

    acc = correct / len(images) * 100
    print(f"\nEnsemble Accuracy: {acc:.2f}%")
    print("Saved CSV & XLSX")




In [5]:
if __name__ == "__main__":
    main()

Loading models...

Ensemble Accuracy: 93.16%
Saved CSV & XLSX


In [6]:
from openpyxl import load_workbook

FILE_PATH = "ensemble_predictions.xlsx"   # change path if needed

# Load workbook
wb = load_workbook(FILE_PATH)
ws = wb.active

# Find "Status" column index
status_col = None
for col in range(1, ws.max_column + 1):
    if ws.cell(row=1, column=col).value == "Status":
        status_col = col
        break

if status_col is None:
    raise ValueError("❌ 'Status' column not found in Excel file")

# Counters
correct = 0
wrong = 0
error = 0

# Read rows
for row in range(2, ws.max_row + 1):
    status = ws.cell(row=row, column=status_col).value

    if status == "Correct":
        correct += 1
    elif status == "Wrong":
        wrong += 1
    else:
        error += 1

# Print summary
print("📊 Ensemble Model Evaluation Summary")
print("----------------------------------")
print(f"Total Samples : {ws.max_row - 1}")
print(f"Correct       : {correct}")
print(f"Wrong         : {wrong}")
print(f"Other/Error   : {error}")

accuracy = (correct / (ws.max_row - 1)) * 100
print(f"\n✅ Accuracy : {accuracy:.2f}%")


📊 Ensemble Model Evaluation Summary
----------------------------------
Total Samples : 1945
Correct       : 1812
Wrong         : 133
Other/Error   : 0

✅ Accuracy : 93.16%
